<a href="https://colab.research.google.com/github/roshan-mahato/PRT_661_WILDFIRE_PREDICTION/blob/main/ml_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Wildfire Fire-Occurrence -Baseline
# Model Training and Evaluation
## Objective

This notebook establishes the **initial machine-learning benchmark** for predicting wildfire occurrence.

Target:

- `label = 1` → fire occurred / qualifying fire detection recorded for the grid-cell/day
- `label = 0` → sampled no-fire grid-cell/day

This is a **binary classification** problem.

**Steps:**
1. Load the final labelled fire-weather dataset.
2. Perform a 60/20/20 stratified train/validation/test split.
3. Train a Dummy baseline, Logistic Regression, and Random Forest.
4. Use the default classification threshold of **0.50**.
5. Compare Accuracy, Precision, Recall, F1-score, and ROC-AUC.


In [ ]:
# Core libraries
from pathlib import Path
import importlib.util
import numpy as np
import pandas as pd

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

import importlib.util
import subprocess
import sys

for package_name, import_name in [('xgboost', 'xgboost'), ('lightgbm', 'lightgbm')]:
    if importlib.util.find_spec(import_name) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package_name])

import xgboost as xgb
import lightgbm as lgb

RANDOM_STATE = 42
pd.set_option('display.max_columns', None)


## 1. Load the labelled dataset

The input file is the CSV created after preprocessing, negative-sample construction, daily aggregation, and fire-weather feature engineering.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
FINAL_CSV = '/content/drive/MyDrive/firms_weather_cleaned_with_negatives.csv'

df = pd.read_csv(FINAL_CSV)

print('Dataset shape:', df.shape)
display(df.head())


Dataset shape: (209793, 22)


,lat_round,lon_round,acq_date,temperature_2m,relative_humidity_2m,wind_speed_10m,wind_gusts_10m,wind_direction_10m,precipitation,soil_moisture_0_to_7cm,vapour_pressure_deficit,et0_fao_evapotranspiration,label,weather_outlier_flag,kbdi,drought_factor,days_since_rain,days_since_cell_start,kbdi_spinup_flag,emc,ffdi,rate_of_spread
0,-43.5,146.0,2024-05-10,12.410222,81.477038,8.248152,17.655999,105.361071,0.0,0.229356,0.312604,0.082951,1,False,0.201304,5.654105,1.0,0.0,True,18.280370,0.782331,0.007510
1,-43.5,146.0,2024-10-21,15.752474,79.673112,9.709290,22.727368,179.126520,0.0,0.243211,0.368014,0.196582,1,False,73.937803,9.655148,165.0,164.0,False,17.448899,1.635698,0.015703
2,-43.5,146.0,2025-08-11,10.022000,72.773119,8.766971,20.448000,98.059778,0.0,0.242200,0.362386,0.078546,1,False,96.476722,10.000000,459.0,458.0,False,15.113268,1.731550,0.016623
3,-43.5,146.5,2024-11-11,8.808064,63.002205,11.854714,36.350769,222.728996,0.0,0.254949,0.451458,0.224982,1,False,0.020848,5.644313,1.0,0.0,True,12.512804,1.423095,0.013662
4,-43.5,146.5,2025-03-19,15.729990,57.477324,7.872270,31.305305,162.541743,0.0,0.273673,0.801754,0.283756,1,True,46.211935,8.150704,129.0,128.0,False,11.355945,2.848787,0.027348


## 2. Basic dataset checks
- `label` exists.
- The target contains only `0` and `1`.
- Both fire and no-fire observations are present.
- Required predictor columns exist.

In [ ]:
assert 'label' in df.columns, "Target column 'label' is missing."

df['label'] = pd.to_numeric(df['label'], errors='coerce')
df = df[df['label'].isin([0, 1])].copy()
df['label'] = df['label'].astype(int)

print('Target distribution:')
print(df['label'].value_counts().sort_index())
print('\nTarget percentage:')
print((df['label'].value_counts(normalize=True).sort_index() * 100).round(2))


Target distribution:
label
0     70521
1    139272
Name: count, dtype: int64

Target percentage:
label
0    33.61
1    66.39
Name: proportion, dtype: float64


## 3. Initial predictor set

In [ ]:
FEATURES = [
    'lat_round',
    'lon_round',
    'temperature_2m',
    'relative_humidity_2m',
    'wind_speed_10m',
    'wind_gusts_10m',
    'wind_direction_10m',
    'precipitation',
    'soil_moisture_0_to_7cm',
    'vapour_pressure_deficit',
    'et0_fao_evapotranspiration',
    'kbdi',
    'drought_factor',
    'days_since_rain',
    'days_since_cell_start',
    'emc',
    'ffdi',
    'rate_of_spread'
]

missing_features = [c for c in FEATURES if c not in df.columns]
assert not missing_features, f'Missing required features: {missing_features}'

X = df[FEATURES].copy()
y = df['label'].copy()

# Convert predictors to numeric and use median values for any remaining missing data.
for col in FEATURES:
    X[col] = pd.to_numeric(X[col], errors='coerce')
    X[col] = X[col].fillna(X[col].median())

print('Number of predictors:', len(FEATURES))
print(FEATURES)


Number of predictors: 18
['lat_round', 'lon_round', 'temperature_2m', 'relative_humidity_2m', 'wind_speed_10m', 'wind_gusts_10m', 'wind_direction_10m', 'precipitation', 'soil_moisture_0_to_7cm', 'vapour_pressure_deficit', 'et0_fao_evapotranspiration', 'kbdi', 'drought_factor', 'days_since_rain', 'days_since_cell_start', 'emc', 'ffdi', 'rate_of_spread']


## 4. Train / validation / test split

A **60/20/20 stratified split** is used.

Stratification keeps approximately the same fire/no-fire proportion in each subset.

- Training set: model fitting
- Validation set: first comparison of the models
- Test set: final baseline check after model selection

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.40,
    stratify=y,
    random_state=RANDOM_STATE
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=RANDOM_STATE
)

print(f'Train:      {len(X_train):,} rows | fire rate = {y_train.mean():.2%}')
print(f'Validation: {len(X_val):,} rows | fire rate = {y_val.mean():.2%}')
print(f'Test:       {len(X_test):,} rows | fire rate = {y_test.mean():.2%}')


Train:      125,875 rows | fire rate = 66.39%
Validation: 41,959 rows | fire rate = 66.38%
Test:       41,959 rows | fire rate = 66.39%


## 5. Baseline models

Three models are compared:

### Dummy Classifier
Provides a minimum benchmark. A useful ML model should clearly outperform this

### Logistic Regression
Provides a simple linear classification baseline. Standardisation is used because Logistic Regression is sensitive to differences in feature scale

### Random Forest
Provides the first non-linear ensemble model. It can capture interactions between weather, drought, and fire-danger variables without requiring feature scaling

All models use the default decision threshold of **0.50**

In [ ]:
models = {
    'Dummy': DummyClassifier(
        strategy='prior',
        random_state=RANDOM_STATE
    ),

    'Logistic Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(
            max_iter=2000,
            random_state=RANDOM_STATE
        ))
    ]),

    'Random Forest': RandomForestClassifier(
        n_estimators=150,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )
}

for name, model in models.items():
    print(f'Training {name}...')
    model.fit(X_train, y_train)

print('Training complete.')


Training Dummy...
Training Logistic Regression...
Training Random Forest...
Training complete.


## 6. Validation-set comparison

In [ ]:
def evaluate_model(model, X_eval, y_eval, threshold=0.50):
    scores = model.predict_proba(X_eval)[:, 1]
    pred = (scores >= threshold).astype(int)

    return {
        'threshold': threshold,
        'accuracy': accuracy_score(y_eval, pred),
        'precision': precision_score(y_eval, pred, zero_division=0),
        'recall': recall_score(y_eval, pred, zero_division=0),
        'f1': f1_score(y_eval, pred, zero_division=0),
        'roc_auc': roc_auc_score(y_eval, scores)
    }

validation_rows = []

for name, model in models.items():
    row = evaluate_model(model, X_val, y_val, threshold=0.50)
    row['model'] = name
    validation_rows.append(row)

validation_results = (
    pd.DataFrame(validation_rows)
    [['model', 'threshold', 'accuracy', 'precision', 'recall', 'f1', 'roc_auc']]
    .sort_values('roc_auc', ascending=False)
    .reset_index(drop=True)
)

display(validation_results.round(4))


,model,threshold,accuracy,precision,recall,f1,roc_auc
0,Random Forest,0.5,0.8368,0.8612,0.899,0.8797,0.9272
1,Logistic Regression,0.5,0.7149,0.7566,0.841,0.7966,0.7831
2,Dummy,0.5,0.6638,0.6638,1.000,0.7980,0.5000


## 7. Select the strongest baseline model
The model with the highest **validation ROC-AUC** is selected

In [ ]:
best_name = validation_results.iloc[0]['model']
best_model = models[best_name]

print('Best baseline model:', best_name)
print('Validation ROC-AUC:', round(validation_results.iloc[0]['roc_auc'], 4))


Best baseline model: Random Forest
Validation ROC-AUC: 0.9272


## 8. Held-out test evaluation

The selected baseline model is evaluated once on the untouched test set using the same default threshold of **0.50**.

In [ ]:
test_metrics = evaluate_model(best_model, X_test, y_test, threshold=0.50)

test_results = pd.DataFrame([
    {'model': best_name, **test_metrics}
])

display(test_results.round(4))

test_scores = best_model.predict_proba(X_test)[:, 1]
test_pred = (test_scores >= 0.50).astype(int)

cm = confusion_matrix(y_test, test_pred)

print('Confusion matrix:')
print(cm)
print('\nRows = actual class, columns = predicted class')


,model,threshold,accuracy,precision,recall,f1,roc_auc
0,Random Forest,0.5,0.8342,0.8581,0.8989,0.878,0.9248


Confusion matrix:
[[ 9963  4141]
 [ 2815 25040]]

Rows = actual class, columns = predicted class


## 8. Additional ensemble models and evaluation

Adding two tree-based boosting models:

- **XGBoost**
- **LightGBM**



In [ ]:
# Add XGBoost and LightGBM.
models['XGBoost'] = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.10,
    subsample=0.80,
    colsample_bytree=1.0,
    eval_metric='logloss',
    random_state=RANDOM_STATE,
    n_jobs=-1,
    tree_method='hist'
)

models['LightGBM'] = lgb.LGBMClassifier(
    n_estimators=200,
    num_leaves=31,
    learning_rate=0.05,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbosity=-1
)

In [ ]:
from sklearn.metrics import balanced_accuracy_score, average_precision_score

def evaluate_model_extended(model, X_eval, y_eval, threshold=0.50):
    scores = model.predict_proba(X_eval)[:, 1]
    pred = (scores >= threshold).astype(int)

    return {
        'threshold': threshold,
        'accuracy': accuracy_score(y_eval, pred),
        'balanced_accuracy': balanced_accuracy_score(y_eval, pred),
        'precision': precision_score(y_eval, pred, zero_division=0),
        'recall': recall_score(y_eval, pred, zero_division=0),
        'f1': f1_score(y_eval, pred, zero_division=0),
        'roc_auc': roc_auc_score(y_eval, scores),
        'pr_auc': average_precision_score(y_eval, scores)
    }


## 9. Models comparision

In [ ]:
for name, model in models.items():
    # Only train models that haven't been trained yet (XGBoost and LightGBM)
    if not hasattr(model, 'coef_') and not hasattr(model, 'feature_importances_') and not isinstance(model, DummyClassifier): # Simple check to avoid re-training already fitted models if they are not pipelines.
        if isinstance(model, Pipeline): # Handle pipelines (Logistic Regression)
            if 'model' in model.named_steps and not hasattr(model.named_steps['model'], 'coef_'):
                print(f'Retraining {name}...')
                model.fit(X_train, y_train)
        else: # Handle direct models (XGBoost, LightGBM, Random Forest)
            print(f'Retraining {name}...')
            model.fit(X_train, y_train)

# Ensure all models are fitted. If the previous loop was not run after adding new models,
# or if we want to be absolutely sure, this block can be run.
# For now, let's just make sure the new models are trained.
for name, model in models.items():
    if name not in validation_results['model'].values:
        print(f'Training {name}...')
        model.fit(X_train, y_train)

comparison_rows = []

for name, model in models.items():
    row = evaluate_model_extended(model, X_val, y_val, threshold=0.50)
    row['model'] = name
    comparison_rows.append(row)

model_comparison = (
    pd.DataFrame(comparison_rows)
    [[
        'model', 'threshold', 'accuracy', 'balanced_accuracy',
        'precision', 'recall', 'f1', 'roc_auc', 'pr_auc'
    ]]
    .sort_values('roc_auc', ascending=False)
    .reset_index(drop=True)
)

display(model_comparison.round(4))

Retraining XGBoost...
Retraining LightGBM...
Training XGBoost...
Training LightGBM...


,model,threshold,accuracy,balanced_accuracy,precision,recall,f1,roc_auc,pr_auc
0,Random Forest,0.5,0.8368,0.8065,0.8612,0.8990,0.8797,0.9272,0.9648
1,XGBoost,0.5,0.8347,0.8096,0.8675,0.8864,0.8769,0.9258,0.9649
2,LightGBM,0.5,0.8291,0.7992,0.8575,0.8905,0.8737,0.9205,0.9622
3,Logistic Regression,0.5,0.7149,0.6534,0.7566,0.8410,0.7966,0.7831,0.8932
4,Dummy,0.5,0.6638,0.5000,0.6638,1.0000,0.7980,0.5000,0.6638
